# EXP: Small Hypergraph Driver Node Selection Methods

In [ ]:
import HAT
import math
import numpy as np
import scipy as sp
import networkx as nx
import random
import networkx as nx
import pandas as pd
import time
from itertools import combinations

def brute_force_min_driver_sets(HG, verbose=False):

    n = HG.nnodes
    all_sys_nodes = set(range(n))

    if verbose:
        print("Start brute-forcing...\n")

    min_size = None
    min_sets = []

    for k in range(1, n+1):
        if verbose:
            print(f"=== Trying driver set size k = {k} ===")

        found_k = []

        for subset in combinations(range(n), k):
            drivers = set(subset)

            # ------------------------------
            # Step 1: dilation check
            # ------------------------------
            if detect_dilation_with_given_drivers(HG, drivers):
                continue
                #if verbose:
                #    print("  skip (dilation) drivers =", sorted(drivers))
                #continue

            # ------------------------------
            # Step 2: accessibility check
            # ------------------------------
            reachable = hypergraph_reachable(HG, drivers)
            reachable_sys = reachable & all_sys_nodes

            if reachable_sys == all_sys_nodes:
                found_k.append(drivers)
                if verbose:
                    print("  Feasible driver set:", sorted(drivers))

        if found_k:
            min_size = k
            min_sets = found_k
            if verbose:
                print(f"\n>>> Minimum driver size = {k}")
                for s in min_sets:
                    print("    ", sorted(s))
            break

    if min_size is None and verbose:
        print("No feasible driver set found: either dilation exists or accessibility cannot be satisfied.")

    return min_size, min_sets

def random_uniform_hypergraph(n, k, m, head_size_range=(1, 3)):
    edge_set = set()
    edge_list = []

    while len(edge_list) < m:
        tail = tuple(sorted(random.sample(range(n), k)))
        remaining = list(set(range(n)) - set(tail))
        hsize = random.randint(head_size_range[0], head_size_range[1])
        hsize = min(hsize, len(remaining)) 
        head = tuple(sorted(random.sample(remaining, hsize)))
        edge = (head, tail)
        if edge not in edge_set:
            edge_set.add(edge)
            edge_list.append({"head": list(head), "tail": list(tail)})
    return edge_list
    
def star_graph(HG):
    G = nx.DiGraph() if HG.directed else nx.Graph()
    system_nodes = [f"n{i}" for i in range(HG.nnodes)]
    G.add_nodes_from(system_nodes, bipartite=0)

    for edge_id in HG.edges.index:
        e_node = f"e{edge_id}"
        G.add_node(e_node, bipartite=1)

        # Get head/tail from HG.edges
        heads = HG.edges.loc[edge_id, "head"]
        tails = HG.edges.loc[edge_id, "tail"]
        heads = [f"n{h}" for h in heads]
        tails = [f"n{t}" for t in tails]

        if HG.directed:
            # tail → edge
            for t in tails:
                G.add_edge(t, e_node)
            # edge → head
            for h in heads:
                G.add_edge(e_node, h)
        else:
            for v in heads + tails:
                G.add_edge(e_node, v)

    return G

def digraph_to_bipartite_H(G):
    H = nx.DiGraph()

    V_plus = {f"{v}+": v for v in G.nodes}   
    V_minus = {f"{v}-": v for v in G.nodes}  
    H.add_nodes_from(V_plus.keys(), bipartite=0)
    H.add_nodes_from(V_minus.keys(), bipartite=1)
    for u, v in G.edges:
        H.add_edge(f"{u}+", f"{v}-")
    return H, V_plus, V_minus

def edge_list_to_incidence(edge_list):
    nodes = set()
    for e in edge_list:
        nodes.update(e["head"])
        nodes.update(e["tail"])    
    nodes = sorted(list(nodes))   
    node_to_idx = {node:i for i,node in enumerate(nodes)}    
    n = len(nodes)
    m = len(edge_list)
    incidence = np.zeros((n, m), dtype=int)    
    for j, e in enumerate(edge_list):
        for h in e["head"]:       # head = +1
            incidence[node_to_idx[h], j] = 1
        for t in e["tail"]:       # tail = -1
            incidence[node_to_idx[t], j] = -1   
    return incidence

def detect_dilation_and_drivers(HG,num_nodes):
    G_star = star_graph(HG)
    G_bip = nx.Graph()
    E_side = set()   
    V_side = set()   
    from networkx.algorithms import bipartite
    for u, v in G_star.edges():
        if isinstance(u, str) and u.startswith("e") \
           and isinstance(v, str) and v.startswith("n"):
            G_bip.add_edge(u, v)
            E_side.add(u)
            V_side.add(v)
    matching = bipartite.hopcroft_karp_matching(G_bip, E_side)
    matched_heads = {
        int(v[1:]) for v in matching
        if isinstance(v, str) and v.startswith("n")
    }
    all_nodes = set(range(HG.nnodes))
    driver_nodes = all_nodes - matched_heads
    dilation_exists = len(matched_heads) < num_nodes
    return dilation_exists, matched_heads, driver_nodes, matching


def detect_dilation_with_given_drivers(HG, drivers):
    G_star = star_graph(HG) 
    from networkx.algorithms import bipartite

    for d in drivers:
        u_name = f"u{d}"
        v_name = f"n{d}"
        if not G_star.has_node(u_name):
            G_star.add_node(u_name)
        G_star.add_edge(u_name, v_name)

    G_bip = nx.Graph()
    E_side = set()
    V_side = set()

    for u, v in G_star.edges():
        if (
            isinstance(u, str) and (u.startswith("e") or u.startswith("u"))
            and isinstance(v, str) and v.startswith("n")
        ):
            G_bip.add_edge(u, v)
            E_side.add(u)
            V_side.add(v)

    matching = bipartite.hopcroft_karp_matching(G_bip, E_side)

    matched_heads = {
        int(v[1:]) for v in matching
        if isinstance(v, str) and v.startswith("n")
    }

    all_nodes = set(range(HG.nnodes))
    unmatched_nodes = all_nodes - matched_heads
    dilation_exists = len(matched_heads) < HG.nnodes

    return dilation_exists


def detect_hyperedge_dilation_N(HG,num_nodes):
    G_star = star_graph(HG)
    H_bip, V_plus, V_minus = digraph_to_bipartite_H(G_star)
    from networkx.algorithms import bipartite
    U = set(V_plus.keys())   # left side = V^+
    matching = bipartite.hopcroft_karp_matching(H_bip, U)
    matched_minus = set(
        v for v in matching
        if isinstance(v, str) and v.endswith("-")
    )
    matched_system_nodes = {V_minus[v] for v in matched_minus}
    all_minus = set(V_minus.keys())
    unmatched_minus = all_minus - matched_minus
    driver_nodes = {V_minus[v] for v in unmatched_minus}
    dilation_exists = len(matched_system_nodes) < num_nodes
    return dilation_exists, matched_system_nodes, driver_nodes, matching


def hypergraph_reachable(HG, start_nodes):
    # extract head/tail
    head_list = HG.edges["head"].tolist()
    tail_list = HG.edges["tail"].tolist()

    accessible = set(start_nodes)
    changed = True

    while changed:
        changed = False
        for h, t in zip(head_list, tail_list):
            if set(t).issubset(accessible):
                before = len(accessible)
                accessible.update(h)
                if len(accessible) > before:
                    changed = True

    return accessible

def add_node_fields(HG):
    HG.nodes["edges"] = [[] for _ in range(len(HG.nodes))]
    HG.nodes["head"]  = [[] for _ in range(len(HG.nodes))]
    HG.nodes["tail"]  = [[] for _ in range(len(HG.nodes))]

    # Create a lookup from node id -> row index in HG.nodes
    node_to_idx = dict(zip(HG.nodes["Nodes"], HG.nodes.index))

    # Iterate over hyperedges
    for _, edge_row in HG.edges.iterrows():
        edge_id = edge_row["Edges"]

        # All nodes in this hyperedge
        for n in edge_row["Nodes"]:
            HG.nodes.at[node_to_idx[n], "edges"].append(edge_id)

        # Head membership
        for n in edge_row["head"]:
            HG.nodes.at[node_to_idx[n], "head"].append(edge_id)

        # Tail membership
        for n in edge_row["tail"]:
            HG.nodes.at[node_to_idx[n], "tail"].append(edge_id)
    return HG

def greedy_accessibility(HG, initial_drivers=[]):
    all_sys_nodes = set(range(HG.nnodes))
    
    # Preprocess hypergraph structure once
    head_list = HG.edges["head"].tolist()
    tail_list = HG.edges["tail"].tolist()
    edge_tails = [set(t) for t in tail_list]
    edge_heads = [set(h) for h in head_list]
    
    # Use precomputed node->edges mapping
    node_edges_list = HG.nodes['edges'].tolist()
    
    # Build lookup: edge -> tail nodes still needed for activation
    edge_missing = [set(tail) for tail in edge_tails]
    
    # Initialize
    current_drivers = set(initial_drivers)
    accessible = set(current_drivers)
    active_edges = set()
    
    # Activate initial edges
    for node in list(accessible):
        for edge_idx in node_edges_list[node]:
            edge_missing[edge_idx].discard(node)
            if not edge_missing[edge_idx] and edge_idx not in active_edges:
                active_edges.add(edge_idx)
                accessible.update(edge_heads[edge_idx])
    
    reachable_sys = accessible & all_sys_nodes
    remaining = all_sys_nodes - reachable_sys
    
    # Greedy loop
    while remaining:
        best_node = None
        best_size = len(remaining)
        best_new_accessible = None
        
        # ====================================================
        # ### NEW: build candidate pool (pruning)
        # ====================================================
        candidate_pool = set()

        for v in remaining:
            for edge_idx in node_edges_list[v]:
                # v is the last missing tail of some inactive edge
                if edge_idx not in active_edges and edge_missing[edge_idx] == {v}:
                    candidate_pool.add(v)
                    break

        # fallback to full remaining set if pruning gives nothing
        if not candidate_pool:
            candidate_pool = remaining
        
        
        for cand in candidate_pool:
            # Calculate incremental impact of adding cand
            new_accessible = accessible | {cand}
            newly_activated = []
            
            # Check only edges that contain cand
            for edge_idx in node_edges_list[cand]:
                if edge_idx not in active_edges:
                    # Check if this edge would become active
                    if edge_missing[edge_idx] <= new_accessible:
                        newly_activated.append(edge_idx)
            
            # Propagate from newly activated edges
            if newly_activated:
                new_accessible = _fast_propagate(
                    new_accessible, newly_activated, active_edges,
                    edge_missing, edge_heads, node_edges_list
                )
            
            new_remaining_size = len(all_sys_nodes - (new_accessible & all_sys_nodes))
            
            if new_remaining_size < best_size:
                best_node = cand
                best_size = new_remaining_size
                best_new_accessible = new_accessible
        
        # Update state with best choice
        current_drivers.add(best_node)
        accessible = best_new_accessible
        
        # Update edge_missing and active_edges for the chosen node
        for edge_idx in node_edges_list[best_node]:
            edge_missing[edge_idx].discard(best_node)
            if not edge_missing[edge_idx] and edge_idx not in active_edges:
                active_edges.add(edge_idx)
        
        remaining = all_sys_nodes - (accessible & all_sys_nodes)
    
    return current_drivers


def _fast_propagate(accessible, newly_activated, already_active, edge_missing, edge_heads, node_edges_list):
    """Propagate reachability using node->edges lookup."""
    result = set(accessible)
    to_process = list(newly_activated)
    processed = set(already_active)
    
    i = 0
    while i < len(to_process):
        edge_idx = to_process[i]
        i += 1
        processed.add(edge_idx)
        
        # Add head nodes from this edge
        new_nodes = edge_heads[edge_idx] - result
        result.update(new_nodes)
        
        # For each new node, check edges it participates in
        for node in new_nodes:
            for check_edge_idx in node_edges_list[node]:
                if check_edge_idx not in processed:
                    # Check if edge becomes active
                    if edge_missing[check_edge_idx].issubset(result):
                        to_process.append(check_edge_idx)
                        processed.add(check_edge_idx)
    
    return result

def run_one_instance(n, k, m, trial_id, verbose=False):
    """
    Run a single trial and return a dictionary with all per-trial results.
    """
    edge_list = random_uniform_hypergraph(n=n, k=k, m=m)
    incidence = edge_list_to_incidence(edge_list)
    nodes = pd.DataFrame({'Nodes': np.arange(n)})
    HG = HAT.Hypergraph(incidence_matrix=incidence, directed=True, nodes=nodes)
    HG = add_node_fields(HG)
    
    # ==========================================================
    # 1. Matching (for dilation detection)
    # ==========================================================
    t0 = time.perf_counter()
    dilation_exists, matched_nodes, driver_nodes, matching = detect_dilation_and_drivers(HG, HG.nnodes)
    t_matching = time.perf_counter() - t0

    if dilation_exists:
        unmatched_nodes = {int(str(u).replace("n","")) for u in driver_nodes}
    else:
        unmatched_nodes = set()
            
    raw_matching_size = len(unmatched_nodes)
    matching_size = max(1, raw_matching_size)

    # ==========================================================
    # 2. Matching + Greedy
    # ==========================================================
    t0 = time.perf_counter()
    drivers_mg = greedy_accessibility(HG, unmatched_nodes)
    t_mg = time.perf_counter() - t0 + t_matching 
    greedy_size = len(drivers_mg)

    # ==========================================================
    # 3. Pure Greedy
    # ==========================================================
    t0 = time.perf_counter()
    drivers_pg = greedy_accessibility(HG, set())
    
    has_dil_pg = detect_dilation_with_given_drivers(HG, drivers_pg)
    if has_dil_pg:
        if verbose:
            print("Warning: pure greedy has dilation.")
        # We still record it — pure greedy is allowed to fail
        pure_greedy_size = np.nan
        t_pg = np.nan
    else:
        pure_greedy_size = len(drivers_pg)
        t_pg = time.perf_counter() - t0
        

    # ==========================================================
    # 4. Optimal brute force (can be VERY slow!)
    # ==========================================================
    t0 = time.perf_counter()
    min_size, min_sets = brute_force_min_driver_sets(HG, verbose=False)
    t_optimal = time.perf_counter() - t0

    # Return a dictionary with all trial-level information
    return {
        'trial_id': trial_id,
        'n': n,
        'k': k,
        'm': m,
        'alpha': m / n,
        'matching_size': matching_size,
        'greedy_size': greedy_size,
        'pure_greedy_size': pure_greedy_size,
        'optimal_size': min_size,
        'time_matching': t_matching,
        'time_mg': t_mg,
        'time_pg': t_pg,
        'time_optimal': t_optimal,
    }


def run_experiment_per_trial(output_file='EXP1_per_trial_v2.csv'):
    """
    Run the experiment and save individual trial results.
    """
    num_trials = 5

    all_results = []
    trial_counter = 0

    for n in [5, 10, 15, 20]:
        for k in [4, 6, 8]:
            for alpha in [0.5, 1]:
                m = int(alpha * n)
                if m > math.comb(n, k):
                    continue

                print(f"Running: n={n}, k={k}, alpha={alpha}, trials={num_trials}")

                for trial in range(num_trials):
                    trial_counter += 1
                    
                    try:
                        result = run_one_instance(n, k, m, trial_id=trial_counter)
                        result['alpha'] = alpha
                        all_results.append(result)
                        
                        # Save after each trial (incremental saving)
                        df = pd.DataFrame(all_results)
                        df.to_csv(output_file, index=False)
                        
                        print(f"  Trial {trial+1}/{num_trials} completed (ID: {trial_counter})")
                        
                    except Exception as e:
                        print(f"  Trial {trial+1}/{num_trials} failed with error: {e}")
                        continue

    print(f"\nExperiment complete! Results saved to {output_file}")
    print(f"Total trials completed: {len(all_results)}")
    
    return pd.DataFrame(all_results)


if __name__ == "__main__":
    # Run the experiment
    df_results = run_experiment_per_trial(output_file='EXP1_per_trial.csv')
    
    # Print summary statistics
    print("\n" + "="*60)
    print("SUMMARY STATISTICS")
    print("="*60)
    
    # Group by parameters and show statistics
    summary = df_results.groupby(['n', 'alpha']).agg({
        'matching_size': ['mean', 'std'],
        'greedy_size': ['mean', 'std'],
        'pure_greedy_size': ['mean', 'std'],
        'optimal_size': ['mean', 'std'],
        'time_matching': ['mean', 'std'],
        'time_mg': ['mean', 'std'],
        'time_pg': ['mean', 'std'],
        'time_optimal': ['mean', 'std'],
    }).round(4)
    
    print(summary)

Running: n=5, k=4, alpha=0.5, trials=5
  Trial 1/5 completed (ID: 1)
  Trial 2/5 completed (ID: 2)
  Trial 3/5 completed (ID: 3)
  Trial 4/5 completed (ID: 4)
  Trial 5/5 completed (ID: 5)
Running: n=5, k=4, alpha=1, trials=5
  Trial 1/5 completed (ID: 6)
  Trial 2/5 completed (ID: 7)
  Trial 3/5 completed (ID: 8)
  Trial 4/5 completed (ID: 9)
  Trial 5/5 completed (ID: 10)
Running: n=10, k=4, alpha=0.5, trials=5
  Trial 1/5 completed (ID: 11)
  Trial 2/5 completed (ID: 12)
  Trial 3/5 completed (ID: 13)
  Trial 4/5 completed (ID: 14)
  Trial 5/5 completed (ID: 15)
Running: n=10, k=4, alpha=1, trials=5
  Trial 1/5 completed (ID: 16)
  Trial 2/5 completed (ID: 17)
  Trial 3/5 completed (ID: 18)
  Trial 4/5 completed (ID: 19)
  Trial 5/5 completed (ID: 20)
Running: n=10, k=6, alpha=0.5, trials=5
  Trial 1/5 completed (ID: 21)
  Trial 2/5 completed (ID: 22)
  Trial 3/5 completed (ID: 23)
  Trial 4/5 completed (ID: 24)
  Trial 5/5 completed (ID: 25)
Running: n=10, k=6, alpha=1, trials=5
  